# 04. 정제표 예시 — 원자료를 "기업 × 분기" 표로 바꿔 보기

**이 노트북이 하는 일:** DART 원자료(보고서 한 건마다 계정 100줄)를 **분석에 쓸 수 있는 표(한 행 = 한 기업의 한 분기)** 로 바꾸는 과정을 한 단계씩 보여 줍니다. 한 기업(디와이덕양)으로 손으로 따라가 본 다음, 28곳 전체에 같은 규칙을 적용합니다.

| 항목 | 내용 |
|---|---|
| 입력 | `data/raw/DART_파일럿/fs/*.json`, `outputs/01_파일럿/파일럿_표본40곳.csv` |
| 출력 | `outputs/01_파일럿/정제표_예시.csv`, `정제표_예시.xlsx` (**예시용**, 아래 `SAVE_FILES=True`일 때만) |
| DART API 호출 | **없음** |
| 실행 시간 | 약 10초 |
| 먼저 알아야 할 것 | 02번(응답의 칸), 03번(계정 찾기, 분기 환원) |
| 원래 스크립트 | `원본_스크립트/04_정제표_예시_원본.py` |

> ### ⚠ 이 표는 "검증 전 예시"입니다
> 분석·논문에 **그대로 쓰면 안 됩니다.** 계정 매핑 규칙이 첫 추정이고, 이자 정의(A3·A16)·차입 계정(A10·A15)·연결/별도 혼재가 아직 결정되지 않았습니다. 이 노트북의 목적은 **"정제가 이런 순서로 이뤄진다"는 것을 이해하는 것**입니다. 회귀나 계수는 계산하지 않습니다.

## 이 노트북의 지도
1. 정제가 무엇인가 — 원자료에서 표로
2. 한 기업 한 해를 손으로 따라가기 (디와이덕양 2022)
3. 규칙을 함수로 만들기 (분기 값 환원, 차입비중, TTM, ICR)
4. 28곳 전체에 적용
5. 결과 표를 읽는 법과 문제점
6. 저장 (선택)

In [1]:
import csv, json, re, sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import config

SAVE_FILES = False        # ← True로 바꾸면 결과를 outputs/01_파일럿에 저장함(기존 정제표_예시.* 파일이 있으면 덮어씀)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 60)

FILE_RE = re.compile(r"^(\d{8})_(\d{4})_(\d{5})_(CFS|OFS)\.json$")
REP_BY_CODE = {v: k for k, v in config.REPORTS.items()}
UNIT = 1_000_000                       # 표에는 백만원 단위로 보여 준다
QUARTER_SOURCE = [(1, "Q1"), (2, "H1"), (3, "Q3"), (4, "FY")]   # 1분기=Q1보고서, 2분기=반기보고서, 3분기=Q3보고서, 4분기=사업보고서
FLOWS = ["revenue", "op_income", "interest_exp", "fin_cost"]      # '기간 동안의 합' 성격: 분기별 값을 만든다
STOCKS = ["assets", "st_borrow", "cur_ltd", "cur_bond"]           # '그 날짜의 값' 성격: 그대로 쓴다
print("흐름(기간 합) 계정:", FLOWS)
print("스냅샷(그 날짜) 계정:", STOCKS)

흐름(기간 합) 계정: ['revenue', 'op_income', 'interest_exp', 'fin_cost']
스냅샷(그 날짜) 계정: ['assets', 'st_borrow', 'cur_ltd', 'cur_bond']


---
# 1. 정제가 무엇인가

**원자료**는 보고서 한 건이 계정 100~120줄이고, 손익 계정에는 **"그 분기 3개월 값"** 과 **"누적 값"** 이 따로 있으며, **4분기 보고서는 없습니다.**

**정제**는 이런 일을 합니다.
1. 100줄 중 필요한 계정 **9개만 골라냄** (계정명 규칙)
2. **분기별 값**을 만듦 — Q1~Q3는 각 보고서의 3개월 값, **Q4 = 연간(사업보고서) − 3분기 누적**
3. 파생 변수 계산 — **차입비중**(= 노출도의 재료), **후행 4분기 합(TTM)**, **ICR**

핵심 구분은 두 가지입니다.
- **흐름(flow)** 계정(매출·영업이익·이자): 기간 동안의 합 → 분기 값으로 쪼개고, 4분기를 합쳐 TTM을 만듦
- **스냅샷(stock)** 계정(총자산·차입금): 그 날짜의 값 → **합치지 않고** 그 분기 말 값을 그대로 씀

---
# 2. 한 기업 한 해를 손으로 따라가기 — 디와이덕양 2022

먼저 네 보고서에서 필요한 계정을 **직접 골라서** 표로 만들어 봅니다.

In [2]:
def parse_amount(text):
    if text is None:
        return None
    s = str(text).replace(",", "").strip()
    if s in ("", "-"):
        return None
    try:
        return float(s)
    except ValueError:
        return None


def ord_key(row):
    try:
        return int(row.get("ord", "0") or 0)
    except ValueError:
        return 0


def find(rows, target):
    # 03번의 first_match와 같은 일: 규칙에 맞는 첫 줄의 이름·당기금액·누적금액을 돌려준다
    sj, pat = config.ACCOUNT_PATTERNS[target]
    sj_re, nm_re = re.compile(sj), re.compile(pat)
    hits = [r for r in rows
            if sj_re.fullmatch(r.get("sj_div", "")) and nm_re.search(re.sub(r"\s+", "", r.get("account_nm", "")))]
    if not hits:
        return None
    hits.sort(key=ord_key)
    r = hits[0]
    return {"name": r.get("account_nm"), "amt": parse_amount(r.get("thstrm_amount")), "cum": parse_amount(r.get("thstrm_add_amount"))}


CORP, YEAR = "00114154", 2022
raw = {}
for rep_name, code in config.REPORTS.items():
    d = json.loads((config.RAW / "fs" / f"{CORP}_{YEAR}_{code}_CFS.json").read_text(encoding="utf-8"))
    raw[rep_name] = d["list"]

rows = []
for t in ["op_income", "fin_cost", "interest_paid_cf", "assets", "st_borrow", "cur_ltd", "cur_bond"]:
    row = {"계정": t}
    for rep_name in ["Q1", "H1", "Q3", "FY"]:
        x = find(raw[rep_name], t)
        row[f"{rep_name} 당기금액"] = None if x is None else x["amt"]
        row[f"{rep_name} 누적"] = None if x is None else x["cum"]
    rows.append(row)
pd.DataFrame(rows).set_index("계정").map(lambda v: None if v is None or pd.isna(v) else f"{v:,.0f}")

,Q1 당기금액,Q1 누적,H1 당기금액,H1 누적,Q3 당기금액,Q3 누적,FY 당기금액,FY 누적
계정,,,,,,,,
op_income,"1,323,150,464","1,323,150,464","2,716,273,613","4,039,424,077","4,249,354,059","8,288,778,136","30,682,700,016",None
fin_cost,"837,050,309","837,050,309","863,430,699","1,700,481,008","923,864,683","2,624,345,691","3,526,755,685",None
interest_paid_cf,"756,780,651",NaN,"1,702,560,442",NaN,"2,622,718,882",NaN,"3,434,859,967",None
assets,"529,823,110,737",NaN,"509,376,240,185",NaN,"527,622,764,099",NaN,"520,217,537,029",None
st_borrow,"39,473,424,000",NaN,"43,938,415,000",NaN,"32,630,060,000",NaN,"43,358,717,000",None
cur_ltd,"10,795,518,108",NaN,"23,911,501,313",NaN,"25,446,558,902",NaN,"23,679,035,761",None
cur_bond,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


### 해석 — 표 읽는 법
- **영업이익·금융원가(손익 계정)**: Q1·반기·Q3 칸에서 `당기금액`은 **그 분기 3개월 값**, `누적`은 **1월부터의 누적**입니다. 사업보고서(FY)의 `당기금액`은 **연간 한 칸**이고 누적 칸은 비어 있습니다(연간 값이 곧 누적).
- **이자지급(현금흐름표)**: `누적` 칸이 비어 있고 `당기금액`에 **누적 값**이 들어 있는 것으로 보입니다(756.8M → 1,702.6M → 2,622.7M → 3,434.9M으로 계속 커짐). 가이드에 현금흐름표 분기 규정이 없어서 "그렇게 보인다"는 관찰입니다.
- **자산총계·차입금(재무상태표)**: 그 분기 **말 날짜의 값**이라 합치지 않습니다.
- **유동성사채**는 이 기업에 없어 표에서 **빈칸**입니다. "정말 0원"인지 "계정이 안 나온 것"인지 이 표만으로는 구분할 수 없습니다(결정 A10).

### 이제 분기 값을 만들어 봅니다
Q1·Q2·Q3는 각 보고서의 `당기금액`(3개월 값)입니다. **Q4는 연간 − 3분기 누적**입니다.

In [3]:
def q_values(target):
    q1 = find(raw["Q1"], target)["amt"]
    q2 = find(raw["H1"], target)["amt"]
    q3 = find(raw["Q3"], target)["amt"]
    q3_cum = find(raw["Q3"], target)["cum"]            # 3분기까지의 누적(1~9월)
    annual = find(raw["FY"], target)["amt"]            # 연간
    q4 = annual - q3_cum                               # 4분기 = 연간 − 3분기 누적
    return [q1, q2, q3, q4], annual, q3_cum

out = []
for t in ["op_income", "fin_cost"]:
    qs, annual, q3_cum = q_values(t)
    out.append({"계정": t, "Q1": qs[0], "Q2": qs[1], "Q3": qs[2], "Q4(계산)": qs[3], "합계": sum(qs), "연간(원문)": annual, "합계-연간": sum(qs) - annual})
pd.DataFrame(out).set_index("계정").map(lambda v: f"{v:,.0f}")

,Q1,Q2,Q3,Q4(계산),합계,연간(원문),합계-연간
계정,,,,,,,
op_income,"1,323,150,464","2,716,273,613","4,249,354,059","22,393,921,880","30,682,700,016","30,682,700,016",0
fin_cost,"837,050,309","863,430,699","923,864,683","902,409,994","3,526,755,685","3,526,755,685",0


### 해석 — 맞는지 검산
- **"합계 − 연간 = 0"** 이면 Q1+Q2+Q3+Q4가 연간과 같다는 뜻입니다. 이 기업은 정확히 0입니다.
- 그런데 이 검산은 **자동으로 맞을 수밖에 없는 구조**입니다. Q4를 "연간 − 3분기 누적"으로 만들었으니 Q1+Q2+Q3 = 3분기 누적일 때 합계는 항상 연간이 됩니다. 진짜 검산은 **"Q1+Q2+Q3 = 3분기 누적(원문)"** 입니다. 03번 노트북에서 한 항등성 점검이 바로 이것입니다(일부 기업에서 어긋남).
- **Q4 영업이익이 Q1~Q3보다 5배 이상 큽니다.** 실제 4분기 실적이 좋았는지, 연말 조정(재고평가·충당금 등) 때문인지 **이 자료만으로는 알 수 없습니다.** 이런 분기별 요동이 있기 때문에 **분기 ICR 대신 4분기 합(TTM)** 을 쓰는 것입니다.

---
# 3. 규칙을 함수로 만들기

위에서 손으로 한 일을 **함수**로 묶어 28곳에 반복합니다. 세 개의 함수입니다.

| 함수 | 하는 일 |
|---|---|
| `load_reports` | 모든 보고서 파일을 읽어 `(기업, 연도, 보고서) → 계정 목록`으로 정리 |
| `build_firm_rows` | 한 기업의 분기별 값을 만듦 (Q1~Q3는 3개월 값, Q4는 연간−3분기 누적) |
| `add_derived` | 차입비중, TTM, ICR 같은 **파생 변수** 계산 |

In [4]:
def load_reports(raw_dir):
    reports, skipped = {}, 0
    for path in sorted(Path(raw_dir).glob("*.json")):
        m = FILE_RE.match(path.name)
        if not m:
            continue
        corp, year, code, fs = m.groups()
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            skipped += 1
            continue
        if data.get("status") != "000":            # 자료 없는 것(013)은 건너뜀 → 그 보고서 값은 '없음'
            continue
        reports[(corp, int(year), REP_BY_CODE.get(code, code))] = (fs, data.get("list", []))
    return reports, skipped


def build_firm_rows(corp, meta, reports):
    recs = []
    years = sorted({y for (c, y, _r) in reports if c == corp})
    for y in years:
        rep = {k: reports.get((corp, y, k)) for k in ("Q1", "H1", "Q3", "FY")}
        fs_set = {v[0] for v in rep.values() if v}
        for qi, rkey in QUARTER_SOURCE:
            if not rep[rkey]:
                continue                          # 그 보고서가 없으면 그 분기는 행을 만들지 않음
            fs, rows = rep[rkey]
            rec = {"corp": corp, "name": meta.get("corp_name", ""), "cls": meta.get("corp_cls", ""),
                   "induty": meta.get("induty_code", ""), "year": y, "q": qi, "fs": fs, "fs_mixed": len(fs_set) > 1}
            for t in STOCKS:                      # 스냅샷: 그 분기말 값 그대로
                x = find(rows, t)
                rec[t] = x["amt"] if x else None
            for t in FLOWS:                       # 흐름: 분기 값
                x = find(rows, t)
                if rkey != "FY":
                    rec[t] = x["amt"] if x else None          # Q1·반기·Q3 보고서의 당기금액 = 3개월 값
                else:
                    annual = x["amt"] if x else None
                    cum = None
                    if rep["Q3"]:
                        x3 = find(rep["Q3"][1], t)
                        cum = x3["cum"] if x3 else None       # 3분기 누적 값이 있으면 그것을 씀
                    if cum is None and all(rep[k] for k in ("Q1", "H1", "Q3")):
                        parts = [find(rep[k][1], t) for k in ("Q1", "H1", "Q3")]
                        if all(p and p["amt"] is not None for p in parts):
                            cum = sum(p["amt"] for p in parts)  # 누적이 없으면 Q1+Q2+Q3 3개월 값의 합으로 대신
                    rec[t] = annual - cum if annual is not None and cum is not None else None   # Q4 = 연간 - 3분기 누적
            recs.append(rec)
    recs.sort(key=lambda r: (r["year"], r["q"]))
    return recs
print("함수 정의 완료")

함수 정의 완료


### 해석 — 코드 읽는 법
- `for qi, rkey in QUARTER_SOURCE`: 분기 1~4를 각각 **Q1보고서, 반기보고서, Q3보고서, 사업보고서**에서 가져옵니다.
- 흐름 계정 처리의 핵심은 두 줄입니다.
  - Q1·반기·Q3: `rec[t] = x["amt"]` — 보고서가 말해 주는 **3개월 값을 그대로**
  - 사업보고서(Q4): `annual - cum` — **연간에서 3분기 누적을 뺀 값**
- **누적이 비어 있으면**(일부 기업) Q1+Q2+Q3의 3개월 값을 더해 3분기 누적을 **대신 만듭니다.** 이때 03번에서 본 "분기 합과 누적이 어긋나는" 경우가 있으면 Q4에 오차가 몰립니다.
- 보고서가 하나라도 없으면 그 분기의 행이 **만들어지지 않거나** 값이 **`None`(없음)** 이 됩니다. **없는 값을 0으로 채우지 않는다**는 것이 중요한 원칙입니다.

In [5]:
def add_derived(recs):
    for i, r in enumerate(recs):
        # (a) 차입비중(%) = (단기차입금 + 유동성장기부채 + 유동성사채) / 자산총계 × 100
        comps = [r["st_borrow"], r["cur_ltd"], r["cur_bond"]]
        r["debt_ratio"], r["debt_note"] = None, ""
        if r["assets"] not in (None, 0):
            if all(c is None for c in comps):
                r["debt_note"] = "차입 관련 세 항목이 모두 없음(0인지 추출 실패인지 구분 불가)"
            else:
                absent = [nm for nm, c in zip(("단기차입금", "유동성장기부채", "유동성사채"), comps) if c is None]
                r["debt_ratio"] = sum(c or 0 for c in comps) / r["assets"] * 100      # 없는 항목은 0으로 계산(주의!)
                r["debt_note"] = ("없는 항목을 0으로 처리: " + ", ".join(absent)) if absent else ""

        # (b) 후행 4분기 합(TTM): 지금 분기와 그 앞 세 분기, 합 4개 분기가 '연속'일 때만 만든다
        window = recs[i - 3: i + 1] if i >= 3 else []
        consecutive = len(window) == 4 and all(
            window[k + 1]["year"] * 4 + window[k + 1]["q"] == window[k]["year"] * 4 + window[k]["q"] + 1 for k in range(3))

        def ttm(key):
            if not consecutive or any(w[key] is None for w in window):
                return None
            return sum(w[key] for w in window)

        r["op_ttm"] = ttm("op_income")
        ie, fc = ttm("interest_exp"), ttm("fin_cost")
        # 이자: 이자비용이 있으면 그것, 없으면 금융원가로 대신(대용)
        r["int_ttm"], r["int_src"] = (ie, "이자비용") if ie is not None else ((fc, "금융비용(대용)") if fc is not None else (None, ""))

        # (c) ICR = TTM 영업이익 / TTM 이자
        r["icr"], r["icr_note"] = None, ""
        if r["op_ttm"] is None or r["int_ttm"] is None:
            r["icr_note"] = "후행 4분기 값이 연속으로 없음 또는 이자 항목 없음"
        elif r["int_ttm"] <= 0:
            r["icr_note"] = "이자 합계가 0 이하라 ICR 정의 불가"
        else:
            r["icr"] = r["op_ttm"] / r["int_ttm"]
            if r["int_src"].startswith("금융비용"):
                r["icr_note"] = "이자비용이 아니라 금융비용으로 계산(환손실 등 포함 가능)"
    return recs
print("함수 정의 완료")

함수 정의 완료


### 해석 — 파생 변수 세 가지가 어떻게 만들어지나
**(a) 차입비중(%)** = (단기차입금 + 유동성장기부채 + 유동성사채) ÷ 자산총계 × 100. 이것이 노출도(E)의 **재료**입니다(진짜 E는 사전기간 평균).
- ⚠ **세 항목 중 일부가 없으면 0으로 계산합니다**(`c or 0`). 정말 0원이면 맞지만, **이름이 달라서 못 찾은 것이면 차입비중이 실제보다 작게** 나옵니다. 그래서 "없는 항목을 0으로 처리" 메모를 남깁니다. 세 항목이 **전부** 없으면 계산하지 않고 "구분 불가"로 표시합니다.

**(b) 후행 4분기 합(TTM)**: 지금 분기와 앞 세 분기를 더합니다. **네 분기가 연속이고 값이 모두 있을 때만** 만듭니다. 연속 판정은 `year*4 + q`로 분기에 번호를 매겨 1씩 증가하는지 봅니다. 분기가 하나라도 비면 TTM은 **없음**이 됩니다(0으로 채우지 않음).

**(c) ICR** = TTM 영업이익 ÷ TTM 이자.
- 이자는 **이자비용이 있으면 그것, 없으면 금융원가(금융비용)** 를 씁니다. 03번에서 본 대로 이자비용 계정은 거의 없어서 **대부분 금융원가로 계산**됩니다. 이 사실이 `이자 출처` 칸에 표시됩니다.
- **이자 합계가 0 이하**이면 ICR을 만들지 않습니다(0으로 나누기 또는 의미 없는 음수).
- **영업이익이 음수여도 ICR은 계산**합니다(음수 ICR). 영업이익이 적자일 때는 이자 부담이 커져도 ICR이 오르는 식으로 **해석이 뒤집힐 수 있습니다**(D6 문제).

---
# 4. 28곳 전체에 적용

In [6]:
reports, skipped = load_reports(config.RAW / "fs")
meta_df = pd.read_csv(config.OUT / "파일럿_표본40곳.csv", dtype=str, keep_default_na=False)
meta = {r["corp_code"]: r for r in meta_df.to_dict("records")}
corps = sorted({c for (c, _y, _r) in reports})
print(f"자료가 있는 보고서 {len(reports)}건, 기업 {len(corps)}곳 (읽지 못한 파일 {skipped}건)")

all_rows = []
for corp in corps:
    recs = add_derived(build_firm_rows(corp, meta.get(corp, {}), reports))
    all_rows.extend(recs)
panel = pd.DataFrame(all_rows)
print("기업×분기 표의 크기:", panel.shape)
panel[["name", "year", "q", "fs", "op_income", "fin_cost", "debt_ratio", "op_ttm", "int_ttm", "icr"]].head(8)

자료가 있는 보고서 651건, 기업 28곳 (읽지 못한 파일 0건)


기업×분기 표의 크기: (651, 23)


,name,year,q,fs,op_income,fin_cost,debt_ratio,op_ttm,int_ttm,icr
0,대륙제관,2018,1,OFS,1.200746e+09,127066094.0,9.873246,NaN,NaN,NaN
1,대륙제관,2018,2,OFS,3.097396e+09,118032187.0,9.574892,NaN,NaN,NaN
2,대륙제관,2018,3,OFS,4.580640e+08,120667836.0,9.681271,NaN,NaN,NaN
3,대륙제관,2018,4,OFS,8.682308e+08,98205582.0,9.714757,5.624436e+09,463971699.0,12.122369
4,대륙제관,2019,1,OFS,7.777830e+08,116072762.0,9.842339,5.201474e+09,452978367.0,11.482830
5,대륙제관,2019,2,OFS,2.816473e+09,120651936.0,9.377241,4.920551e+09,455598116.0,10.800200
6,대륙제관,2019,3,OFS,4.262202e+08,103498748.0,9.049063,4.888707e+09,438429028.0,11.150509
7,대륙제관,2019,4,OFS,4.340102e+09,148689511.0,8.997103,8.360578e+09,488912957.0,17.100339


### 해석
- 28곳의 모든 분기를 합쳐 **한 행 = 한 기업의 한 분기**인 표가 만들어졌습니다. 행 수는 기업마다 자료가 있는 분기 수의 합입니다.
- `NaN`은 값이 **없음**입니다. 처음 몇 분기에서 `op_ttm`·`icr`이 `NaN`인 것은 **앞 세 분기가 없어서 4분기 합을 못 만들었기** 때문입니다(정상 동작).

In [7]:
# 계산 가능 여부를 요약한다 (분석 결과가 아니라 '구멍이 어디 있는지' 보는 표)
summary = []
for corp in corps:
    g = panel[panel["corp"] == corp]
    summary.append({
        "기업": g["name"].iloc[0], "자료 있는 분기": len(g),
        "이자비용 있는 분기": int(g["interest_exp"].notna().sum()),
        "금융비용 있는 분기": int(g["fin_cost"].notna().sum()),
        "차입비중 계산 가능": int(g["debt_ratio"].notna().sum()),
        "ICR 계산 가능": int(g["icr"].notna().sum()),
        "연결·별도 혼재 연도 수": int(g.loc[g["fs_mixed"], "year"].nunique()),
    })
sm = pd.DataFrame(summary)
sm

,기업,자료 있는 분기,이자비용 있는 분기,금융비용 있는 분기,차입비중 계산 가능,ICR 계산 가능,연결·별도 혼재 연도 수
0,대륙제관,28,0,28,22,25,0
1,디와이덕양,28,0,28,27,25,0
2,TYM,28,0,28,24,25,0
3,삼보산업,28,0,28,22,25,0
4,삼화전기,28,0,28,24,25,0
5,차AI헬스케어,28,0,28,27,25,1
6,한탑,28,0,28,0,25,0
7,영보화학,28,0,28,22,25,0
8,에이프로젠,28,0,28,28,25,0
9,한일홀딩스,28,0,28,28,25,0


In [8]:
print("기업 28곳 중:")
print(f"  '이자비용' 계정이 한 번이라도 있는 기업: {(sm['이자비용 있는 분기'] > 0).sum()}곳")
print(f"  '금융비용(금융원가)'이 있는 기업: {(sm['금융비용 있는 분기'] > 0).sum()}곳")
print(f"  ICR을 한 번이라도 계산할 수 있는 기업: {(sm['ICR 계산 가능'] > 0).sum()}곳")
print()
print(f"전체 {len(panel)}행 중")
print(f"  차입 세 항목이 모두 없어 차입비중을 계산 못 한 행: {panel['debt_note'].str.startswith('차입 관련 세 항목').sum()}행")
print(f"  없는 항목을 0으로 처리해 계산한 행: {panel['debt_note'].str.startswith('없는 항목을 0').sum()}행")
print(f"  ICR이 계산된 행: {panel['icr'].notna().sum()}행")
print(f"  그중 이자를 금융비용(대용)으로 계산한 행: {(panel['icr'].notna() & panel['int_src'].str.startswith('금융')).sum()}행")

기업 28곳 중:
  '이자비용' 계정이 한 번이라도 있는 기업: 1곳
  '금융비용(금융원가)'이 있는 기업: 26곳
  ICR을 한 번이라도 계산할 수 있는 기업: 26곳

전체 651행 중
  차입 세 항목이 모두 없어 차입비중을 계산 못 한 행: 172행
  없는 항목을 0으로 처리해 계산한 행: 468행
  ICR이 계산된 행: 560행
  그중 이자를 금융비용(대용)으로 계산한 행: 560행


In [9]:
# '없는 항목을 0으로 처리'한 행에서, 어떤 항목이 없었나?
zero = panel.loc[panel["debt_note"].str.startswith("없는 항목을 0"), "debt_note"].str.replace("없는 항목을 0으로 처리: ", "", regex=False)
print("없는 항목의 조합별 행 수:")
print(zero.value_counts().to_string())

없는 항목의 조합별 행 수:
debt_note
유동성장기부채, 유동성사채    228
유동성사채             226
단기차입금, 유동성사채        7
단기차입금               4
유동성장기부채             3


### 해석
- **이자비용 계정이 있는 기업은 1곳뿐**이고, 금융비용이 있는 기업은 28곳 중 26곳입니다(03번과 일치). **ICR이 계산된 560행 전부가 "이자비용이 아니라 금융비용으로 계산"한 값**입니다. 이 표의 ICR은 엄밀히 **"영업이익 ÷ 금융원가"** 입니다.
- **차입 관련 세 항목이 모두 없어 차입비중을 계산하지 못한 행은 172개(651행 중 26%)** 입니다. **일부 항목이 없어서 0으로 처리한 행은 468개(72%)** 입니다. 즉 **세 항목이 모두 찾아진 행은 11개뿐**입니다.
- 위 조합 표를 보면 0 처리 468행 중 **유동성사채가 없는 행이 461개**(226+228+7)로 거의 전부입니다(03번에서 확보율 3%). 사채를 발행하지 않은 기업이 많으므로 이 경우의 "0 처리"는 대체로 합리적일 가능성이 있습니다.
- 눈에 띄는 것은 **유동성장기부채가 없다고 처리된 행이 231개**(228+3)로 많다는 점입니다. 장기차입금이 없는 기업일 수도 있지만 이름이 달라서 못 찾았을 수도 있습니다(03번에서 확보율 40%).
- **단기차입금이 없다고 처리된 행은 11개**(7+4)뿐입니다. 이 항목이 노출도(E)의 가장 큰 구성요소일 것으로 보이므로, 이 11행이 진짜 0인지가 우선 확인 대상입니다.
- 어느 경우든 "없음"을 한꺼번에 0으로 처리하고 있으므로 **진짜 0인지 이름 불일치인지 구분하지 못합니다.** 이것이 A10의 쟁점입니다.
- **"ICR 계산 가능" 분기 수는 "자료 있는 분기"보다 항상 적습니다.** TTM을 만들려면 연속 4분기가 필요해서 처음 3분기는 무조건 비고, 중간에 보고서가 빠지면 그 구간도 비기 때문입니다.

---
# 5. 한 기업의 결과 표를 읽어 보기 — 디와이덕양 2022년

TTM과 ICR이 위에서 손으로 계산한 값과 같은지 확인합니다.

In [10]:
g = panel[(panel["corp"] == CORP) & (panel["year"] >= 2021)].copy()
view = g[["year", "q", "op_income", "fin_cost", "op_ttm", "int_ttm", "icr", "assets", "st_borrow", "cur_ltd", "debt_ratio", "int_src"]].copy()
for c in ["op_income", "fin_cost", "op_ttm", "int_ttm", "assets", "st_borrow", "cur_ltd"]:
    view[c] = (view[c] / UNIT).round(1)
view["icr"] = view["icr"].round(2)
view["debt_ratio"] = view["debt_ratio"].round(2)
view = view.rename(columns={"op_income": "영업이익(분기,백만)", "fin_cost": "금융원가(분기,백만)", "op_ttm": "영업이익 TTM", "int_ttm": "이자 TTM",
                            "icr": "ICR(배)", "assets": "자산", "st_borrow": "단기차입금", "cur_ltd": "유동성장기", "debt_ratio": "차입비중(%)"})
view

,year,q,"영업이익(분기,백만)","금융원가(분기,백만)",영업이익 TTM,이자 TTM,ICR(배),자산,단기차입금,유동성장기,차입비중(%),int_src
40,2021,1,771.8,804.7,4386.0,3471.3,1.26,501483.8,58341.4,14204.9,14.47,금융비용(대용)
41,2021,2,1589.0,838.5,8849.2,3387.1,2.61,501600.0,43817.6,13614.5,11.45,금융비용(대용)
42,2021,3,254.1,781.6,7935.9,3299.0,2.41,491796.4,45341.3,13302.3,11.92,금융비용(대용)
43,2021,4,3646.1,1104.9,6261.1,3529.7,1.77,523564.3,41725.9,11894.1,10.24,금융비용(대용)
44,2022,1,1323.2,837.1,6812.4,3562.0,1.91,529823.1,39473.4,10795.5,9.49,금융비용(대용)
45,2022,2,2716.3,863.4,7939.7,3587.0,2.21,509376.2,43938.4,23911.5,13.32,금융비용(대용)
46,2022,3,4249.4,923.9,11934.9,3729.2,3.20,527622.8,32630.1,25446.6,11.01,금융비용(대용)
47,2022,4,22393.9,902.4,30682.7,3526.8,8.70,520217.5,43358.7,23679.0,12.89,금융비용(대용)
48,2023,1,8801.7,1027.2,38161.3,3716.9,10.27,552451.5,23215.8,22540.6,8.28,금융비용(대용)
49,2023,2,1949.6,1137.7,37394.6,3991.2,9.37,577803.1,31741.5,10423.9,7.30,금융비용(대용)


In [11]:
row = g[(g["year"] == 2022) & (g["q"] == 4)].iloc[0]
print("2022 Q4 행의 TTM ICR:", round(row["icr"], 4))
print("손으로 계산한 값 30,682,700,016 ÷ 3,526,755,685 =", round(30682700016 / 3526755685, 4))
print("같은가:", abs(row["icr"] - 30682700016 / 3526755685) < 1e-9)
print()
print(f"2022 Q4 차입비중: {row['debt_ratio']:.2f}%  (= (단기차입금 {row['st_borrow']:,.0f} + 유동성장기 {row['cur_ltd']:,.0f}) ÷ 자산 {row['assets']:,.0f})")

2022 Q4 행의 TTM ICR: 8.7
손으로 계산한 값 30,682,700,016 ÷ 3,526,755,685 = 8.7
같은가: True

2022 Q4 차입비중: 12.89%  (= (단기차입금 43,358,717,000 + 유동성장기 23,679,035,761) ÷ 자산 520,217,537,029)


### 해석
- **2022 Q4의 TTM ICR이 손으로 계산한 값(30,682,700,016 ÷ 3,526,755,685)과 같습니다.** 이는 2022년 4분기 4개를 합한 값이 사업보고서의 연간 값과 같아서 생기는 일치이며, 함수가 의도대로 동작한다는 확인입니다. (이 일치가 자료가 옳다는 뜻은 아닙니다.)
- **TTM은 분기마다 한 칸씩 밀리는 4분기 창**입니다. 2022 Q3의 TTM은 2021 Q4~2022 Q3의 합입니다. 그래서 2021년 자료까지 받아야 2022 Q1~Q3의 ICR을 만들 수 있고, 이것이 수집 연도를 2018년부터 하는 이유입니다.
- 차입비중은 **그 분기 말의 값**이라 분기마다 달라집니다. 실제 노출도 E는 이 값의 **사전기간(2019Q1~2021Q2) 평균**입니다. 이 노트북은 아직 E를 만들지 않았습니다.
- 이 기업은 **금융원가가 이자지급과 거의 같고(비율 0.97)** 항등식도 정확히 맞는 **비교적 깨끗한 사례**입니다. 다른 기업들은 03번에서 본 것처럼 훨씬 지저분합니다.

---
# 6. 이 표의 문제점 정리

| 문제 | 어디서 드러나나 | 표에 미치는 영향 | 결정 |
|---|---|---|---|
| ICR 분모가 이자비용이 아니라 **금융원가** | `이자 출처` 칸 | ICR이 금융비용 보상비율이 됨. 환율 영향이 섞일 수 있음 | A3·A16 |
| 차입 항목 **부재를 0으로 처리** | `차입비중 비고` 칸 | 차입비중(노출도)이 과소 계산될 수 있음 | A10·A15 |
| 차입 세 항목이 **전부 없는** 분기 | 같은 칸 | 차입비중이 비어 있음 | A10 |
| **연결·별도 혼재** | `fs_mixed` 칸 | 한 기업의 숫자가 해에 따라 규모가 달라짐 | 사전 고정 문서 3절 |
| 3분기 누적과 분기 합의 **불일치** | 03번 | Q4 값에 오차가 몰림 | A13 |
| 영업이익 **음수** | 영업이익 TTM | ICR 해석이 뒤집힘 | A4, D6 |
| 사전기간에 **분기가 빈** 기업 | ICR 계산 가능 분기 수 | TTM이 비어 분석에서 빠짐 | A5, A7 |

### 이 노트북의 한계
- 계정 규칙(`ACCOUNT_PATTERNS`)이 **첫 추정**입니다. 변형 이름을 넣으면 표가 바뀝니다.
- **28곳, 대표성 없음.** 이 표의 어떤 요약도 모집단에 일반화할 수 없습니다.
- **정정공시**가 반영되었는지 확인하지 않았습니다.
- 단위를 "원"이라고 가정했으나 기업마다 같은지는 전수 확인하지 않았습니다(03번 참고).
- 계수·회귀·유의성은 **계산하지 않았습니다.**

In [12]:
# 선택: 결과를 파일로 저장한다. SAVE_FILES=True일 때만 실행된다.
if SAVE_FILES:
    out_csv = config.OUT / "정제표_예시.csv"
    out_xlsx = config.OUT / "정제표_예시.xlsx"
    panel.to_csv(out_csv, index=False, encoding="utf-8-sig")
    with pd.ExcelWriter(out_xlsx) as w:
        panel.to_excel(w, sheet_name="정제표_예시", index=False)
        sm.to_excel(w, sheet_name="점검_요약", index=False)
    print("저장:", out_csv.name, out_xlsx.name)
else:
    print("SAVE_FILES=False → 파일을 저장하지 않았습니다.")

SAVE_FILES=False → 파일을 저장하지 않았습니다.


---
# 7. 다음 단계

이 노트북은 **정제의 흐름을 이해하기 위한 예시**입니다. 본 수집이 끝나면 정제 코드를 **검증된 규칙으로 다시** 만들어야 하고, 그 전에 다음 결정이 필요합니다.

1. **이자 정의** (A3·A16) — 금융원가를 쓸지, 주석에서 이자비용을 따로 가져올지
2. **차입 계정 매핑** (A15)과 **부재 처리** (A10)
3. **연결/별도 혼재** 처리 규칙
4. **상장폐지 기업** 처리 (A9·A14)

그다음 ① 선별 스크립트 ② 본 수집 스크립트 ③ 노출도 E(사전기간 평균) 계산 ④ 진단 6종 순서로 진행합니다. 회귀는 그 뒤입니다.